# Ellipsoid Fitting via Least Squares with Ellipsoid-Specific Constraints
## Reproducible Workflow — Li & Griffiths (2004)

This notebook demonstrates the Python implementation of the ellipsoid-specific least-squares fitting
algorithm described in:

> Li, Q. and Griffiths, J. G. (2004).  
> *Least squares ellipsoid specific fitting.*  
> Proceedings of the Geometric Modeling and Processing, 2004. IEEE, pp. 335–340.  
> DOI: [10.1109/GMAP.2004.1290055](https://doi.org/10.1109/GMAP.2004.1290055)

### Contents
1. [Install & import](#1-install--import)
2. [Synthetic data generation](#2-synthetic-data-generation)
3. [Fitting an axis-aligned ellipsoid](#3-fitting-an-axis-aligned-ellipsoid)
4. [Fitting a rotated ellipsoid](#4-fitting-a-rotated-ellipsoid)
5. [Fitting from CSV datasets](#5-fitting-from-csv-datasets)
6. [Accuracy vs noise study](#6-accuracy-vs-noise-study)
7. [Visualisation](#7-visualisation)

## 1. Install & import

In [ ]:
# If running from the repository root, the package is importable directly.
# Otherwise: pip install -e .
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import numpy as np
import matplotlib
matplotlib.use("Agg")  # headless backend for CI / notebook export
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from ellipsoid_fitting import (
    fit_ellipsoid,
    generate_ellipsoid_points,
    algebraic_distance,
    residuals_rms,
)
print("ellipsoid_fitting imported successfully")

## 2. Synthetic data generation

The `generate_ellipsoid_points` function creates quasi-uniformly distributed points on an ellipsoid surface using the Fibonacci sphere algorithm, optionally perturbed with Gaussian noise.

In [ ]:
TRUE_CENTRE = (1.0, 2.0, 3.0)
TRUE_RADII  = (5.0, 3.0, 2.0)

pts = generate_ellipsoid_points(
    centre=TRUE_CENTRE,
    radii=TRUE_RADII,
    n_points=300,
    noise_std=0.05,
    seed=42,
)

print(f"Shape : {pts.shape}")
print(f"x range: [{pts[:,0].min():.2f}, {pts[:,0].max():.2f}]")
print(f"y range: [{pts[:,1].min():.2f}, {pts[:,1].max():.2f}]")
print(f"z range: [{pts[:,2].min():.2f}, {pts[:,2].max():.2f}]")

## 3. Fitting an axis-aligned ellipsoid

In [ ]:
x, y, z = pts[:, 0], pts[:, 1], pts[:, 2]
result = fit_ellipsoid(x, y, z)

print("=" * 55)
print("Li–Griffiths Ellipsoid Fit")
print("=" * 55)
print(f"True  centre : {np.array(TRUE_CENTRE)}")
print(f"Fitted centre: {result['centre'].round(4)}")
print()
print(f"True  radii  : {np.sort(np.array(TRUE_RADII))[::-1]}")
print(f"Fitted radii : {result['radii'].round(4)}")
print()
print(f"RMS algebraic residual: {residuals_rms(x, y, z, result):.6f}")
print("=" * 55)

## 4. Fitting a rotated ellipsoid

The algorithm handles arbitrary orientations without modification.

In [ ]:
from scipy.spatial.transform import Rotation

R = Rotation.from_euler('xyz', [30, 45, 60], degrees=True).as_matrix()
pts_rot = generate_ellipsoid_points(
    centre=(0.0, 0.0, 0.0),
    radii=(6.0, 4.0, 2.5),
    rotation=R,
    n_points=400,
    noise_std=0.10,
    seed=7,
)

result_rot = fit_ellipsoid(pts_rot[:, 0], pts_rot[:, 1], pts_rot[:, 2])
print(f"True  radii : {np.sort([6.0, 4.0, 2.5])[::-1]}")
print(f"Fitted radii: {result_rot['radii'].round(3)}")
print(f"Centre      : {result_rot['centre'].round(3)}")
print(f"RMS residual: {residuals_rms(pts_rot[:,0], pts_rot[:,1], pts_rot[:,2], result_rot):.4f}")

## 5. Fitting from CSV datasets

In [ ]:
data_dir = os.path.join(os.getcwd(), "..", "data")

for fname in sorted(os.listdir(data_dir)):
    if fname.endswith(".csv"):
        path = os.path.join(data_dir, fname)
        data = np.loadtxt(path, delimiter=",", skiprows=1)
        r = fit_ellipsoid(data[:,0], data[:,1], data[:,2])
        rms = residuals_rms(data[:,0], data[:,1], data[:,2], r)
        print(f"{fname}")
        print(f"  centre : {r['centre'].round(3)}")
        print(f"  radii  : {r['radii'].round(3)}")
        print(f"  RMS    : {rms:.5f}")
        print()

## 6. Accuracy vs noise study

How does centre-estimation error grow with increasing noise?

In [ ]:
noise_levels = np.linspace(0.0, 0.5, 20)
centre_errors = []
radii_errors  = []

TRUE_C = np.array([0.0, 0.0, 0.0])
TRUE_R = np.array([4.0, 3.0, 2.0])

for sigma in noise_levels:
    p = generate_ellipsoid_points(
        centre=TRUE_C, radii=TRUE_R,
        n_points=400, noise_std=sigma, seed=0,
    )
    try:
        res = fit_ellipsoid(p[:,0], p[:,1], p[:,2])
        centre_errors.append(np.linalg.norm(res['centre'] - TRUE_C))
        radii_errors.append(np.linalg.norm(np.sort(res['radii'])[::-1] - np.sort(TRUE_R)[::-1]))
    except ValueError:
        centre_errors.append(np.nan)
        radii_errors.append(np.nan)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(noise_levels, centre_errors, 'o-')
axes[0].set_xlabel("Noise σ"); axes[0].set_ylabel("Centre error (L2)")
axes[0].set_title("Centre recovery accuracy")
axes[1].plot(noise_levels, radii_errors, 's-', color='tab:orange')
axes[1].set_xlabel("Noise σ"); axes[1].set_ylabel("Radii error (L2)")
axes[1].set_title("Semi-axis recovery accuracy")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("noise_study.png", dpi=120)
plt.show()
print("Figure saved to noise_study.png")

## 7. Visualisation

Draw the fitted ellipsoid surface over the noisy data.

In [ ]:
pts_vis = generate_ellipsoid_points(
    centre=(1.0, 2.0, 3.0), radii=(5.0, 3.0, 2.0),
    n_points=300, noise_std=0.05, seed=42,
)
res_vis = fit_ellipsoid(pts_vis[:,0], pts_vis[:,1], pts_vis[:,2])

cx, cy, cz = res_vis['centre']
r1, r2, r3 = res_vis['radii']
axes_mat   = res_vis['axes']

u = np.linspace(0, 2*np.pi, 60)
v = np.linspace(0, np.pi, 30)
xs = r1 * np.outer(np.cos(u), np.sin(v))
ys = r2 * np.outer(np.sin(u), np.sin(v))
zs = r3 * np.outer(np.ones_like(u), np.cos(v))

shape = xs.shape
pts_m = np.column_stack([xs.ravel(), ys.ravel(), zs.ravel()]) @ axes_mat.T
xs2 = pts_m[:,0].reshape(shape) + cx
ys2 = pts_m[:,1].reshape(shape) + cy
zs2 = pts_m[:,2].reshape(shape) + cz

fig = plt.figure(figsize=(9, 7))
ax  = fig.add_subplot(111, projection='3d')
ax.scatter(pts_vis[:,0], pts_vis[:,1], pts_vis[:,2], s=4, alpha=0.4, label='Noisy data')
ax.plot_surface(xs2, ys2, zs2, alpha=0.18, color='orange')
ax.set_title("Li–Griffiths Ellipsoid Fit")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax.legend()
plt.tight_layout()
plt.savefig("ellipsoid_fit_vis.png", dpi=120)
plt.show()
print("Figure saved to ellipsoid_fit_vis.png")